In [1]:
import pandas as pd
import numpy as np
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [2]:
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\docto\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\docto\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\docto\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [6]:
df = pd.read_csv("../data/processed/email_spam_cleaned.csv")

df.head()

,title,text,type,email_length
0,?? the secrets to SUCCESS,"Hi James,\n\nHave you claim your complimentary...",spam,302
1,?? You Earned 500 GCLoot Points,"\nalt_text\nCongratulations, you just earned\n...",not spam,350
2,?? Your GitHub launch code,"Here's your GitHub launch code, @Mortyj420!\n ...",not spam,166
3,[The Virtual Reward Center] Re: ** Clarifications,"Hello,\n \nThank you for contacting the Virtua...",not spam,399
4,"10-1 MLB Expert Inside, Plus Everything You Ne...","Hey Prachanda Rawal,\n\nToday's newsletter is ...",spam,6079


In [7]:
print("Dataset Shape:", df.shape)

df.info()

Dataset Shape: (83, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 83 entries, 0 to 82
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         83 non-null     object
 1   text          83 non-null     object
 2   type          83 non-null     object
 3   email_length  83 non-null     int64 
dtypes: int64(1), object(3)
memory usage: 2.7+ KB


In [8]:
df["email"] = (
    df["title"].fillna("") +
    " " +
    df["text"].fillna("")
)

df[["email","type"]].head()

,email,type
0,"?? the secrets to SUCCESS Hi James,\n\nHave yo...",spam
1,?? You Earned 500 GCLoot Points \nalt_text\nCo...,not spam
2,?? Your GitHub launch code Here's your GitHub ...,not spam
3,[The Virtual Reward Center] Re: ** Clarificati...,not spam
4,"10-1 MLB Expert Inside, Plus Everything You Ne...",spam


In [9]:
df["label"] = df["type"].map({
    "spam":1,
    "not spam":0
})

df[["type","label"]].head()

,type,label
0,spam,1
1,not spam,0
2,not spam,0
3,not spam,0
4,spam,1


In [10]:
stop_words = set(stopwords.words("english"))

lemmatizer = WordNetLemmatizer()

In [11]:
def clean_text(text):

    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove email addresses
    text = re.sub(r"\S+@\S+", "", text)

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    words = text.split()

    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)

In [12]:
df["clean_email"] = df["email"].apply(clean_text)

df[["email","clean_email"]].head()

,email,clean_email
0,"?? the secrets to SUCCESS Hi James,\n\nHave yo...",secret success hi james claim complimentary gi...
1,?? You Earned 500 GCLoot Points \nalt_text\nCo...,earned gcloot point alttext congratulation ear...
2,?? Your GitHub launch code Here's your GitHub ...,github launch code here github launch code mor...
3,[The Virtual Reward Center] Re: ** Clarificati...,virtual reward center clarification hello than...
4,"10-1 MLB Expert Inside, Plus Everything You Ne...",mlb expert inside plus everything need blockbu...


In [13]:
df = df[df["clean_email"].str.strip() != ""]

print(df.shape)

(83, 7)


In [14]:
df.head()

,title,text,type,email_length,email,label,clean_email
0,?? the secrets to SUCCESS,"Hi James,\n\nHave you claim your complimentary...",spam,302,"?? the secrets to SUCCESS Hi James,\n\nHave yo...",1,secret success hi james claim complimentary gi...
1,?? You Earned 500 GCLoot Points,"\nalt_text\nCongratulations, you just earned\n...",not spam,350,?? You Earned 500 GCLoot Points \nalt_text\nCo...,0,earned gcloot point alttext congratulation ear...
2,?? Your GitHub launch code,"Here's your GitHub launch code, @Mortyj420!\n ...",not spam,166,?? Your GitHub launch code Here's your GitHub ...,0,github launch code here github launch code mor...
3,[The Virtual Reward Center] Re: ** Clarifications,"Hello,\n \nThank you for contacting the Virtua...",not spam,399,[The Virtual Reward Center] Re: ** Clarificati...,0,virtual reward center clarification hello than...
4,"10-1 MLB Expert Inside, Plus Everything You Ne...","Hey Prachanda Rawal,\n\nToday's newsletter is ...",spam,6079,"10-1 MLB Expert Inside, Plus Everything You Ne...",1,mlb expert inside plus everything need blockbu...


In [15]:
df.to_csv("../data/email_spam_cleaned.csv", index=False)

print("Preprocessed dataset saved successfully!")

Preprocessed dataset saved successfully!


In [16]:
df = df.drop(columns=["email_length"])

In [17]:
df.to_csv("../data/processed/email_spam_cleaned.csv", index=False)

In [18]:
df.head()

,title,text,type,email,label,clean_email
0,?? the secrets to SUCCESS,"Hi James,\n\nHave you claim your complimentary...",spam,"?? the secrets to SUCCESS Hi James,\n\nHave yo...",1,secret success hi james claim complimentary gi...
1,?? You Earned 500 GCLoot Points,"\nalt_text\nCongratulations, you just earned\n...",not spam,?? You Earned 500 GCLoot Points \nalt_text\nCo...,0,earned gcloot point alttext congratulation ear...
2,?? Your GitHub launch code,"Here's your GitHub launch code, @Mortyj420!\n ...",not spam,?? Your GitHub launch code Here's your GitHub ...,0,github launch code here github launch code mor...
3,[The Virtual Reward Center] Re: ** Clarifications,"Hello,\n \nThank you for contacting the Virtua...",not spam,[The Virtual Reward Center] Re: ** Clarificati...,0,virtual reward center clarification hello than...
4,"10-1 MLB Expert Inside, Plus Everything You Ne...","Hey Prachanda Rawal,\n\nToday's newsletter is ...",spam,"10-1 MLB Expert Inside, Plus Everything You Ne...",1,mlb expert inside plus everything need blockbu...


FINAL DATASET PREPROCESSING 

In [1]:
import pandas as pd
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\docto\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\docto\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\docto\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [3]:
df = pd.read_csv(
    "X:/AI-SPAM-PHISHING-DETECTOR/data/processed/final_dataset.csv",
    keep_default_na=False
)

df.head()

,title,text,type
0,?? the secrets to SUCCESS,"Hi James,\n\nHave you claim your complimentary...",spam
1,?? You Earned 500 GCLoot Points,"\nalt_text\nCongratulations, you just earned\n...",not spam
2,?? Your GitHub launch code,"Here's your GitHub launch code, @Mortyj420!\n ...",not spam
3,[The Virtual Reward Center] Re: ** Clarifications,"Hello,\n \nThank you for contacting the Virtua...",not spam
4,"10-1 MLB Expert Inside, Plus Everything You Ne...","Hey Prachanda Rawal,\n\nToday's newsletter is ...",spam


In [4]:
df["clean_text"] = df["text"]

df.head()

,title,text,type,clean_text
0,?? the secrets to SUCCESS,"Hi James,\n\nHave you claim your complimentary...",spam,"Hi James,\n\nHave you claim your complimentary..."
1,?? You Earned 500 GCLoot Points,"\nalt_text\nCongratulations, you just earned\n...",not spam,"\nalt_text\nCongratulations, you just earned\n..."
2,?? Your GitHub launch code,"Here's your GitHub launch code, @Mortyj420!\n ...",not spam,"Here's your GitHub launch code, @Mortyj420!\n ..."
3,[The Virtual Reward Center] Re: ** Clarifications,"Hello,\n \nThank you for contacting the Virtua...",not spam,"Hello,\n \nThank you for contacting the Virtua..."
4,"10-1 MLB Expert Inside, Plus Everything You Ne...","Hey Prachanda Rawal,\n\nToday's newsletter is ...",spam,"Hey Prachanda Rawal,\n\nToday's newsletter is ..."


In [5]:
df["clean_text"] = df["clean_text"].str.lower()

In [6]:
def remove_html(text):
    return re.sub(r"<.*?>", "", text)

df["clean_text"] = df["clean_text"].apply(remove_html)

In [7]:
def remove_urls(text):
    return re.sub(r"http\S+|www\S+", "", text)

df["clean_text"] = df["clean_text"].apply(remove_urls)

In [8]:
def remove_email(text):
    return re.sub(r"\S+@\S+", "", text)

df["clean_text"] = df["clean_text"].apply(remove_email)

In [9]:
def remove_numbers(text):
    return re.sub(r"\d+", "", text)

df["clean_text"] = df["clean_text"].apply(remove_numbers)

In [10]:
def remove_punctuation(text):
    return text.translate(
        str.maketrans("", "", string.punctuation)
    )

df["clean_text"] = df["clean_text"].apply(remove_punctuation)

In [11]:
df["clean_text"] = df["clean_text"].str.strip()

df["clean_text"] = df["clean_text"].str.replace(
    r"\s+",
    " ",
    regex=True
)

In [12]:
stop_words = set(stopwords.words("english"))

def remove_stopwords(text):
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

df["clean_text"] = df["clean_text"].apply(remove_stopwords)

In [13]:
lemmatizer = WordNetLemmatizer()

def lemmatize(text):
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words]
    return " ".join(words)

df["clean_text"] = df["clean_text"].apply(lemmatize)

In [14]:
df[["text", "clean_text"]].head(10)

,text,clean_text
0,"Hi James,\n\nHave you claim your complimentary...",hi james claim complimentary gift yet ive comp...
1,"\nalt_text\nCongratulations, you just earned\n...",alttext congratulation earned completed follow...
2,"Here's your GitHub launch code, @Mortyj420!\n ...",here github launch code mortyj octocat standin...
3,"Hello,\n \nThank you for contacting the Virtua...",hello thank contacting virtual reward center v...
4,"Hey Prachanda Rawal,\n\nToday's newsletter is ...",hey prachanda rawal today newsletter jampacked...
5,Model Casting Call\nThank you for taking the t...,model casting call thank taking time register ...
6,Today more than ever you need to upskill and r...,today ever need upskill reskill global job mar...
7,"\nLogo Image\nSenol Yildirim,\n\nSomeone signe...",logo image senol yildirim someone signedin acc...
8,"Hi,\n\n \n\nThank you for your interest in joi...",hi thank interest joining appen email official...
9,GOOD DAY SIR/MADAM \n\n In 2020 I applied to t...,good day sirmadam applied institution didnt kn...


In [15]:
df.to_csv(
    "X:/AI-SPAM-PHISHING-DETECTOR/data/processed/final_dataset_cleaned.csv",
    index=False
)

print("✅ Cleaned dataset saved successfully!")

✅ Cleaned dataset saved successfully!
